In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import re
import string
from datasets import Dataset
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import torch
from sklearn.metrics import  precision_recall_fscore_support

In [ ]:
fake=pd.read_csv('Fake.csv')
true=pd.read_csv('True.csv')

In [ ]:
fake['class']=0
true['class']=1

In [ ]:
data=pd.concat([fake,true],axis=0)

In [ ]:
data=data.drop(['title','subject','date'],axis=1)

In [ ]:
data.reset_index(inplace=True)

In [ ]:
data.drop(['index'],axis=1,inplace=True)

In [ ]:
def clean(text):
    text=text.lower()
    text=re.sub(r"\[.*?\]","",text)
    text=re.sub(r"\\W"," ",text)
    text=re.sub(r"https?://\S+|www\.\S+","",text)
    text=re.sub(r"<.*?>+","",text)
    text=re.sub(r"[%s]" % re.escape(string.punctuation),"",text)
    text=re.sub(r"\n","",text)
    text=re.sub(r"\w*\d\w*","",text)
    return text


In [ ]:
data['text']=data['text'].apply(clean)

In [ ]:
x=data['text']
y=data['class']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=42)

In [ ]:
train_df = pd.DataFrame({'text': x_train, 'label': y_train})
test_df = pd.DataFrame({'text': x_test, 'label': y_test})

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)


In [ ]:
pip install torch transformers  flask langdetect

In [ ]:

model_name = "xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)



Map:   0%|          | 0/33673 [00:00<?, ? examples/s]

Map:   0%|          | 0/11225 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:

model = XLMRobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Fake News", 1: "Real News"},
    label2id={"Fake News": 0, "Real News": 1}
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# training_args = TrainingArguments(
#     output_dir="./results",
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     logging_steps=20,
#     load_best_model_at_end=True,
#     report_to="none"
# )


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16, # Lower to 8 if you run into CUDA Out-Of-Memory errors
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1",     # Selects best model based on F1 score
    greater_is_better=True,
    save_total_limit=2,             # Saves hard drive space
    fp16=torch.cuda.is_available(), # Accelerates training on GPU
    report_to="none"
)


In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)


    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 8. Run Training
print("Starting training process...")
trainer.train()

Starting training process...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000134,0.009918,0.998753,0.998754,0.998753,0.998753
2,0.000512,0.003077,0.999287,0.999288,0.999287,0.999287
3,0.000039,0.001894,0.999733,0.999733,0.999733,0.999733


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6315, training_loss=0.012069403386386502, metrics={'train_runtime': 1695.334, 'train_samples_per_second': 59.586, 'train_steps_per_second': 3.725, 'total_flos': 1.328960785070592e+16, 'train_loss': 0.012069403386386502, 'epoch': 3.0})

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create a folder path inside your Drive
DRIVE_SAVE_PATH = "/content/drive/MyDrive/fake news detection with multi language"


Mounted at /content/drive


In [ ]:

# 3. Save model and tokenizer directly to Drive
trainer.save_model(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)

print(f"Model permanently saved to Google Drive at: {DRIVE_SAVE_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model permanently saved to Google Drive at: /content/drive/MyDrive/fake news detection with multi language
